In [ ]:
# ============================================================
# Career Guidance System - Model Training Script
# Loads career_recommendation.csv, trains and compares four
# classification models (KNN, Decision Tree, Random Forest,
# XGBoost), and saves the best-performing model along with its
# label binarizer and input encoders.
# ============================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

# --- Step 1: Load dataset ---
career = pd.read_csv("career_recommendation.csv")
print("Loaded:", career.shape)

# --- Step 2: Clean ---
career.drop_duplicates(inplace=True)
career.fillna(0, inplace=True)

# --- Step 3: Encode categorical inputs (THREE SEPARATE encoders,
#             fit directly on the ORIGINAL STRING COLUMNS — this is
#             the critical part that kept breaking) ---
gender_encoder = LabelEncoder()
degree_encoder = LabelEncoder()
field_encoder = LabelEncoder()

career['gender'] = gender_encoder.fit_transform(career['gender'])
career['degree_level'] = degree_encoder.fit_transform(career['degree_level'])
career['field_of_study'] = field_encoder.fit_transform(career['field_of_study'])

# Self-check — MUST print text values, not numbers. If this prints
# numbers instead of words, STOP and do not proceed further.
print("gender classes:", list(gender_encoder.classes_))
print("degree classes:", list(degree_encoder.classes_))
print("field classes:", list(field_encoder.classes_))

# --- Step 4: Feature matrix ---
feature_cols = [
    'age', 'gender', 'degree_level', 'field_of_study', 'gpa', 'years_experience',
    'python', 'java', 'c_cpp', 'sql', 'machine_learning', 'data_analysis',
    'cloud_computing', 'cybersecurity', 'web_development', 'devops', 'networking',
    'communication', 'leadership', 'problem_solving', 'teamwork', 'adaptability',
]
X = career[feature_cols]

# --- Step 5: Targets (multi-label Top-3) ---
y_raw = career[['recommended_job_1', 'recommended_job_2', 'recommended_job_3']].values.tolist()
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(y_raw)
print("Career classes:", list(mlb.classes_))

# --- Step 6: Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# --- Step 7: Train and compare models ---
models = {
    "KNN": OneVsRestClassifier(KNeighborsClassifier(n_neighbors=5)),
    "Decision Tree": OneVsRestClassifier(DecisionTreeClassifier(max_depth=10, random_state=42)),
    "Random Forest": OneVsRestClassifier(RandomForestClassifier(n_estimators=200, random_state=42)),
    "XGBoost": OneVsRestClassifier(XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42, eval_metric='logloss'
    )),
}

best_model = None
best_name = None
best_f1 = 0

for name, model in models.items():
    print("=" * 50)
    print(name)
    print("=" * 50)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='micro')
    print(classification_report(y_test, y_pred))
    print(f"{name} micro F1: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_name = name

print()
print(f"BEST MODEL: {best_name} (F1 = {best_f1:.4f})")

# --- Step 8: Export the 3 artifacts ---
joblib.dump(best_model, "career_model.pkl")
joblib.dump(mlb, "label_binarizer.pkl")
joblib.dump(
    {"gender": gender_encoder, "degree_level": degree_encoder, "field_of_study": field_encoder},
    "input_encoders.pkl"
)

print()
print("DONE. 3 files created: career_model.pkl, label_binarizer.pkl, input_encoders.pkl")
print("Download them from the Files panel on the left.")

Loaded: (2000, 25)
gender classes: ['Female', 'Male']
degree classes: ['Bachelor', 'Master', 'PhD']
field classes: ['AI', 'Computer Science', 'Cybersecurity', 'Data Science', 'Software Engineering']
Career classes: ['AI Researcher', 'Backend Developer', 'Business Intelligence Analyst', 'Cloud Engineer', 'Cybersecurity Analyst', 'Data Analyst', 'Data Scientist', 'DevOps Engineer', 'Frontend Developer', 'Full Stack Developer', 'Machine Learning Engineer', 'Security Engineer', 'Software Engineer']
KNN


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.82      0.16      0.27        56
           1       0.23      0.08      0.12        74
           2       0.33      0.21      0.26       101
           3       0.61      0.54      0.57       177
           4       0.40      0.17      0.24       108
           5       0.00      0.00      0.00        10
           6       0.14      0.02      0.04        41
           7       0.58      0.43      0.50       175
           8       0.00      0.00      0.00        27
           9       0.00      0.00      0.00         7
          10       0.38      0.21      0.27       107
          11       0.00      0.00      0.00        21
          12       0.76      0.83      0.80       296

   micro avg       0.60      0.41      0.49      1200
   macro avg       0.33      0.21      0.24      1200
weighted avg       0.52      0.41      0.44      1200
 samples avg       0.60      0.41      0.47      1200

KNN micro F1: 0.4887
Deci

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.89      0.70      0.78        56
           1       1.00      0.38      0.55        74
           2       0.97      0.92      0.94       101
           3       0.95      0.99      0.97       177
           4       0.95      0.94      0.94       108
           5       1.00      0.20      0.33        10
           6       1.00      0.46      0.63        41
           7       0.95      1.00      0.97       175
           8       1.00      0.26      0.41        27
           9       0.00      0.00      0.00         7
          10       1.00      0.82      0.90       107
          11       1.00      0.05      0.09        21
          12       0.97      1.00      0.98       296

   micro avg       0.96      0.85      0.90      1200
   macro avg       0.90      0.59      0.65      1200
weighted avg       0.96      0.85      0.88      1200
 samples avg       0.97      0.85      0.89      1200

Random Forest micro F1: 0

In [ ]:
import joblib
e = joblib.load('input_encoders.pkl')
print(e['gender'].classes_)
print(e['degree_level'].classes_)
print(e['field_of_study'].classes_)

['Female' 'Male']
['Bachelor' 'Master' 'PhD']
['AI' 'Computer Science' 'Cybersecurity' 'Data Science'
 'Software Engineering']
